# TruePrice Combined YOLO Training: Fruit + Camel Doll

목표: 과일 detector와 `camel_doll` detector를 **하나의 12-class YOLO 모델**로 재학습합니다.

최종 class order는 backend와 동일하게 고정합니다.

```text
0 tomato
1 apple
2 avocado
3 blueberry
4 cherry
5 kiwi
6 mango
7 orange
8 rockmelon
9 strawberry
10 cherry_tomato
11 camel_doll
```

중요: `camel_doll_yolo_bootstrap` 단독 데이터셋은 class id가 `0`입니다. 과일 데이터셋에 그대로 섞으면 `camel_doll`이 `tomato`로 학습됩니다. 따라서 반드시 병합 과정에서 `camel_doll: 0 -> 11`로 remap해야 합니다. 이 노트북은 `ml/scripts/train_trueprice_yolo.py --preset combined_mvp`가 만든 병합 데이터셋을 학습합니다.

## 0. Local 준비: 병합 데이터셋 zip 만들기

Colab에 오기 전에 Mac 터미널에서 아래를 실행하세요.

```bash
cd "/Users/shyoon840/HGU/3-1/HCI/TeamProject/hci_222"

# 1) 과일 + cherry_tomato + camel_doll 데이터를 class id remap해서 병합
python ml/scripts/train_trueprice_yolo.py \
  --preset combined_mvp \
  --prepare-only \
  --allow-empty-classes

# 2) 구조 검증
python ml/tools/yolo_dataset_audit.py dataset_sources/trueprice_yolo_bootstrap

# 3) Colab 업로드용 zip 생성
zip -r trueprice_yolo_combined.zip dataset_sources/trueprice_yolo_bootstrap
```

Colab에서는 `trueprice_yolo_combined.zip`만 업로드하면 됩니다.

In [ ]:
# 1. Runtime check
import os
import sys
from pathlib import Path

try:
    import torch
    print('CUDA available:', torch.cuda.is_available())
    if torch.cuda.is_available():
        print('GPU:', torch.cuda.get_device_name(0))
    else:
        print('⚠️ Colab Runtime > Change runtime type > GPU 로 변경하세요.')
except Exception as exc:
    print('torch import failed:', exc)

print('Python:', sys.version)
print('Working dir:', os.getcwd())

In [ ]:
# 2. Install dependencies
!pip -q install ultralytics pyyaml

from ultralytics import YOLO
import yaml
print('Ultralytics ready')

## 3. Dataset 업로드

아래 셀을 실행해서 로컬에서 만든 `trueprice_yolo_combined.zip`을 업로드하세요.

이미 Google Drive에 올려둔 경우에는 업로드 셀을 건너뛰고 `DATASET_ZIP`만 Drive 경로로 지정하면 됩니다.

In [ ]:
# 3A. Browser upload
from google.colab import files

uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

In [ ]:
# 3B. Resolve dataset zip path
from pathlib import Path

# Browser upload를 쓴 경우 첫 번째 zip 파일을 자동 선택합니다.
zip_candidates = sorted(Path('/content').glob('*.zip'))
if not zip_candidates:
    raise FileNotFoundError('No zip file found in /content. Upload trueprice_yolo_combined.zip first.')

DATASET_ZIP = str(zip_candidates[0])
print('DATASET_ZIP =', DATASET_ZIP)

# Drive를 쓰는 경우 위 자동 선택 대신 아래처럼 직접 지정하세요.
# DATASET_ZIP = '/content/drive/MyDrive/trueprice/trueprice_yolo_combined.zip'

In [ ]:
# 4. Unzip dataset and fix data.yaml path for Colab
from pathlib import Path
import shutil
import yaml

DATASET_ROOT = Path('/content/trueprice_yolo_data')
if DATASET_ROOT.exists():
    shutil.rmtree(DATASET_ROOT)
DATASET_ROOT.mkdir(parents=True, exist_ok=True)

!unzip -q "$DATASET_ZIP" -d "$DATASET_ROOT"

# zip 내부가 dataset_sources/trueprice_yolo_bootstrap 형태인지 탐색
candidates = list(DATASET_ROOT.rglob('data.yaml'))
if not candidates:
    raise FileNotFoundError('data.yaml not found after unzip')

DATA_YAML = candidates[0]
YOLO_ROOT = DATA_YAML.parent
print('YOLO_ROOT =', YOLO_ROOT)
print('DATA_YAML =', DATA_YAML)

with open(DATA_YAML, 'r') as f:
    data = yaml.safe_load(f)

data['path'] = str(YOLO_ROOT)
data['train'] = 'train/images'
data['val'] = 'valid/images'
data['test'] = 'test/images'

with open(DATA_YAML, 'w') as f:
    yaml.safe_dump(data, f, sort_keys=False, allow_unicode=True)

print(Path(DATA_YAML).read_text())

In [ ]:
# 5. Dataset audit: split/image/label/class counts
from pathlib import Path
from collections import Counter
import yaml

with open(DATA_YAML, 'r') as f:
    data_cfg = yaml.safe_load(f)

names = data_cfg['names']
if isinstance(names, dict):
    class_names = {int(k): v for k, v in names.items()}
else:
    class_names = {i: name for i, name in enumerate(names)}

print('Classes:')
for i, name in class_names.items():
    print(f'{i:2d}: {name}')

all_counts = Counter()
for split in ['train', 'valid', 'test']:
    image_dir = YOLO_ROOT / split / 'images'
    label_dir = YOLO_ROOT / split / 'labels'
    images = [p for p in image_dir.glob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.webp'}]
    labels = list(label_dir.glob('*.txt'))
    split_counts = Counter()
    errors = []

    image_stems = {p.stem for p in images}
    label_stems = {p.stem for p in labels}
    for stem in sorted(label_stems - image_stems)[:10]:
        errors.append(f'label without image: {stem}')
    for stem in sorted(image_stems - label_stems)[:10]:
        errors.append(f'image without label: {stem}')

    for label_path in labels:
        for line_no, raw in enumerate(label_path.read_text().splitlines(), 1):
            raw = raw.strip()
            if not raw:
                continue
            parts = raw.split()
            if len(parts) != 5:
                errors.append(f'{label_path.name}:{line_no} invalid field count')
                continue
            cls = int(parts[0])
            vals = [float(x) for x in parts[1:]]
            if cls not in class_names:
                errors.append(f'{label_path.name}:{line_no} unknown class {cls}')
                continue
            if any(v < 0 or v > 1 for v in vals):
                errors.append(f'{label_path.name}:{line_no} coord out of range')
            split_counts[cls] += 1
            all_counts[cls] += 1

    print('\nSplit:', split)
    print(' images:', len(images), 'labels:', len(labels), 'boxes:', sum(split_counts.values()))
    if errors:
        print(' errors:', errors[:20])
    for cls, name in class_names.items():
        print(f'  {cls:2d} {name:14s}: {split_counts[cls]}')

print('\nTotal boxes by class:')
for cls, name in class_names.items():
    print(f'{cls:2d} {name:14s}: {all_counts[cls]}')

empty = [name for cls, name in class_names.items() if all_counts[cls] == 0]
if empty:
    print('\n⚠️ Empty classes:', empty)
    print('해당 class는 학습되지 않습니다. cherry_tomato/camel_doll label 상태를 확인하세요.')
else:
    print('\nNo empty classes.')

## 6. Train

MVP 기준 추천값:

- `yolo11n.pt`: 빠른 MVP용
- `epochs=80`: Colab T4 기준 현실적인 시작점
- `imgsz=640`
- `batch=16`: OOM 발생 시 8로 낮추기

낙타인형 데이터가 과일보다 적으면, 추후에는 class imbalance 보완이 필요합니다. 우선은 단일 모델 동작 검증을 목표로 갑니다.

In [ ]:
# 6. Train combined 12-class model
from ultralytics import YOLO

MODEL_NAME = 'yolo11n.pt'
EPOCHS = 80
IMG_SIZE = 640
BATCH = 16
RUN_NAME = 'trueprice_fruit_camel_yolo11n'

model = YOLO(MODEL_NAME)

results = model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH,
    patience=20,
    seed=222,
    workers=2,
    project='/content/runs',
    name=RUN_NAME,
    exist_ok=True,
)

In [ ]:
# 7. Validate best model
from ultralytics import YOLO
from pathlib import Path

BEST_PT = Path('/content/runs') / RUN_NAME / 'weights' / 'best.pt'
print('BEST_PT:', BEST_PT)
if not BEST_PT.exists():
    raise FileNotFoundError(BEST_PT)

best_model = YOLO(str(BEST_PT))
metrics = best_model.val(data=str(DATA_YAML), imgsz=640)
print(metrics)

In [ ]:
# 8. Quick inference smoke test on a few validation images
from pathlib import Path
from IPython.display import display

valid_images = sorted((YOLO_ROOT / 'valid' / 'images').glob('*'))[:6]
print('Sample images:', len(valid_images))

for image_path in valid_images:
    result = best_model.predict(str(image_path), imgsz=640, conf=0.25, save=True, project='/content/predict', name='smoke', exist_ok=True)
    print(image_path.name, result[0].boxes.cls.cpu().numpy() if result[0].boxes is not None else [])

print('Predictions saved under /content/predict/smoke')

## 9. Export / Download

Flutter/FastAPI MVP에서는 우선 PyTorch `best.pt`를 backend에서 사용합니다.

최종 배치 위치:

```text
backend/models/best.pt
```

`.env`:

```env
TRUEPRICE_DETECTOR_MODE=yolo
TRUEPRICE_YOLO_MODEL_PATH=backend/models/best.pt
```

In [ ]:
# 9A. Download best.pt
from google.colab import files

files.download(str(BEST_PT))

In [ ]:
# 9B. Optional exports
# ONNX export
# best_model.export(format='onnx', imgsz=640)

# TFLite export - can take longer and may install extra dependencies
# best_model.export(format='tflite', imgsz=640)

## 10. Local 적용 후 검증

Colab에서 받은 `best.pt`를 로컬에 배치합니다.

```bash
cd "/Users/shyoon840/HGU/3-1/HCI/TeamProject/hci_222"
mkdir -p backend/models
cp ~/Downloads/best.pt backend/models/best.pt
```

`.env` 확인:

```env
TRUEPRICE_DETECTOR_MODE=yolo
TRUEPRICE_YOLO_MODEL_PATH=backend/models/best.pt
# 단일 통합 모델을 쓰면 extra model은 비워두거나 제거
# TRUEPRICE_YOLO_EXTRA_MODEL_PATHS=
```

Backend smoke test:

```bash
cd backend
python -m pytest tests/test_scan_api.py
uvicorn app.main:app --host 0.0.0.0 --port 8000
```

주의: 기존 `best_camel_doll_only.pt` 병렬 추론은 임시 전략입니다. 이 노트북으로 만든 통합 모델이 안정화되면 단일 `best.pt`만 사용하는 쪽이 맞습니다.